In [ ]:
!pip uninstall -y onnxruntime onnxruntime-gpu

!pip install -q \
onnxruntime-gpu==1.22.0 \
insightface==0.7.3

In [ ]:
import onnxruntime as ort
print(ort.get_available_providers())

In [ ]:
-q optuna

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from enum import Enum

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import cv2

import optuna

from insightface.app import FaceAnalysis

SEED = 42

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)

In [ ]:
# UTKFace
UTK_DATASET_PATH = "/kaggle/input/datasets/jangedoo/utkface-new/UTKFace"

# Kaggle dataset
KAGGLE_DATASET_PATH = "/kaggle/input/datasets/aadyasingh55/face-age-gender-dataset/face_dataset"
KAGGLE_TRAIN_CSV = "/kaggle/input/datasets/aadyasingh55/face-age-gender-dataset/face_dataset/train.csv"

IMG_SIZE = 112
EMBED_DIM = 512

BATCH_SIZE = 256
EPOCHS = 10

AGE_LABELS = [
    "<18",
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65+"
]

NUM_AGE = len(AGE_LABELS)
NUM_GENDER = 2

In [ ]:
def age_to_label(age):

    if age < 18:
        return 0

    elif age <= 24:
        return 1

    elif age <= 34:
        return 2

    elif age <= 44:
        return 3

    elif age <= 54:
        return 4

    elif age <= 64:
        return 5

    else:
        return 6

In [ ]:
def load_utkface_dataframe(dataset_path):

    records = []

    for fname in os.listdir(dataset_path):

        if not fname.endswith(".jpg"):
            continue

        try:
            age, gender = fname.split("_")[:2]

            age = int(age)
            gender = int(gender)

            if gender not in [0, 1]:
                continue

            records.append({
                "image_path": os.path.join(dataset_path, fname),
                "age_label": age_to_label(age),
                "gender_label": gender
            })

        except:
            continue

    return pd.DataFrame(records)

utk_df = load_utkface_dataframe(UTK_DATASET_PATH)

print("UTK:", len(utk_df))

In [ ]:
def load_kaggle_dataframe(csv_path, root_dir):

    df = pd.read_csv(csv_path)

    df["image_path"] = df["full_path"].apply(
        lambda x: os.path.join(root_dir, x)
    )

    df["age_label"] = df["age"].apply(age_to_label)

    df["gender_label"] = df["gender"].astype(int)

    return df[["image_path", "age_label", "gender_label"]]

kaggle_df = load_kaggle_dataframe(
    KAGGLE_TRAIN_CSV,
    KAGGLE_DATASET_PATH
)

print("Kaggle:", len(kaggle_df))

In [ ]:
combined_df = pd.concat(
    [utk_df, kaggle_df],
    ignore_index=True
)

combined_df = combined_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

train_df, test_df = train_test_split(
    combined_df,
    test_size=0.15,
    stratify=combined_df["age_label"],
    random_state=SEED
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["age_label"],
    random_state=SEED
)

print(len(train_df), len(val_df), len(test_df))

In [ ]:
import onnxruntime as ort

print(ort.get_available_providers())

In [ ]:
from insightface.app import FaceAnalysis

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider"]
)

app.prepare(ctx_id=0)

In [ ]:
def extract_embedding(img_path):

    img = cv2.imread(img_path)

    if img is None:
        return None

    img = cv2.resize(img, (640, 640))  # IMPORTANT SPEED + STABILITY

    faces = app.get(img)

    if len(faces) == 0:
        return None

    return faces[0].embedding

In [ ]:
def build_embeddings(df):

    embeddings = []
    age_labels = []
    gender_labels = []

    for row in tqdm(df.itertuples(index=False), total=len(df)):

        emb = extract_embedding(row.image_path)

        if emb is None:
            continue

        embeddings.append(emb)
        age_labels.append(row.age_label)
        gender_labels.append(row.gender_label)

    embeddings = np.stack(embeddings).astype(np.float32)
    age_labels = np.array(age_labels, dtype=np.int64)
    gender_labels = np.array(gender_labels, dtype=np.int64)

    return embeddings, age_labels, gender_labels

In [ ]:
, y_age_train, y_gender_train = build_embeddings(train_df)

X_val, y_age_val, y_gender_val = build_embeddings(val_df)

X_test, y_age_test, y_gender_test = build_embeddings(test_df)

In [ ]:
import numpy as np

np.savez(
    "train_embeddings.npz",
    X=X_train,
    y_age=y_age_train,
    y_gender=y_gender_train
)

np.savez(
    "val_embeddings.npz",
    X=X_val,
    y_age=y_age_val,
    y_gender=y_gender_val
)

np.savez(
    "test_embeddings.npz",
    X=X_test,
    y_age=y_age_test,
    y_gender=y_gender_test
)

print("Saved all embeddings successfully")

In [ ]:
import torch
import torch.nn as nn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class MultiTaskMLP(nn.Module):

    def __init__(self, hidden_dim=512, dropout=0.3):

        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(512, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.age_head = nn.Linear(hidden_dim, NUM_AGE)
        self.gender_head = nn.Linear(hidden_dim, NUM_GENDER)

    def forward(self, x):

        x = self.shared(x)

        age_logits = self.age_head(x)
        gender_logits = self.gender_head(x)

        return age_logits, gender_logits

In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader


class EmbeddingDataset(Dataset):

    def __init__(self, X, y_age, y_gender):

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_age = torch.tensor(y_age, dtype=torch.long)
        self.y_gender = torch.tensor(y_gender, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_age[idx], self.y_gender[idx]


def load_npz(path):

    data = np.load(path)

    return data["X"], data["y_age"], data["y_gender"]

In [ ]:
, y_age_train, y_gender_train = load_npz("train_embeddings.npz")
X_val, y_age_val, y_gender_val = load_npz("val_embeddings.npz")

In [ ]:
import torch.nn.functional as F


def train_model(model, train_loader, val_loader, lr=1e-3, weight_decay=1e-4):

    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    age_loss_fn = nn.CrossEntropyLoss()
    gender_loss_fn = nn.CrossEntropyLoss()

    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(5):  # keep small for Optuna speed

        model.train()

        for x, y_age, y_gender in train_loader:

            x = x.to(DEVICE)
            y_age = y_age.to(DEVICE)
            y_gender = y_gender.to(DEVICE)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast():

                age_logits, gender_logits = model(x)

                loss_age = age_loss_fn(age_logits, y_age)
                loss_gender = gender_loss_fn(gender_logits, y_gender)

                loss = 0.7 * loss_age + 0.3 * loss_gender

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    # validation score
    model.eval()

    correct_age = 0
    correct_gender = 0
    total = 0

    with torch.no_grad():
        for x, y_age, y_gender in val_loader:

            x = x.to(DEVICE)

            age_logits, gender_logits = model(x)

            pred_age = age_logits.argmax(1).cpu()
            # pred_gender = gender_logits.argmax(1).cpu()
            gender_probs = torch.softmax(gender_logits, dim=1)

            gender_confidence, pred_gender = gender_probs.max(1)

            pred_gender = pred_gender.cpu()
            gender_confidence = gender_confidence.cpu()

            correct_age += (pred_age == y_age).sum().item()
            correct_gender += (pred_gender == y_gender).sum().item()

            total += y_age.size(0)

    age_acc = correct_age / total
    gender_acc = correct_gender / total

    return 0.7 * age_acc + 0.3 * gender_acc

In [ ]:
import optuna


def objective(trial):

    hidden_dim = trial.suggest_categorical("hidden_dim", [256, 512, 768])
    dropout = trial.suggest_float("dropout", 0.2, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    model = MultiTaskMLP(hidden_dim=hidden_dim, dropout=dropout)

    train_dataset = EmbeddingDataset(X_train, y_age_train, y_gender_train)
    val_dataset = EmbeddingDataset(X_val, y_age_val, y_gender_val)

    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256)

    score = train_model(
        model,
        train_loader,
        val_loader,
        lr=lr,
        weight_decay=weight_decay
    )

    return score

In [ ]:
study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=10)  # KEEP SMALL FOR SPEED

print("Best params:", study.best_params)
print("Best score:", study.best_value)

In [ ]:
best = study.best_params

model = MultiTaskMLP(
    hidden_dim=best["hidden_dim"],
    dropout=best["dropout"]
)

In [ ]:
model = MultiTaskMLP(
    hidden_dim=best["hidden_dim"],
    dropout=best["dropout"]
).to(DEVICE)

In [ ]:
import numpy as np
from torch.utils.data import DataLoader

train = np.load("train_embeddings.npz")
val = np.load("val_embeddings.npz")

X_train, y_age_train, y_gender_train = train["X"], train["y_age"], train["y_gender"]
X_val, y_age_val, y_gender_val = val["X"], val["y_age"], val["y_gender"]

In [ ]:
train_loader = DataLoader(
    EmbeddingDataset(X_train, y_age_train, y_gender_train),
    batch_size=256,
    shuffle=True
)

val_loader = DataLoader(
    EmbeddingDataset(X_val, y_age_val, y_gender_val),
    batch_size=256,
    shuffle=False
)

In [ ]:
import torch
import torch.nn as nn

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=best["lr"],
    weight_decay=best["weight_decay"]
)

age_loss_fn = nn.CrossEntropyLoss()
gender_loss_fn = nn.CrossEntropyLoss()

In [ ]:
scaler = torch.amp.GradScaler("cuda")

EPOCHS = 25  # FINAL TRAINING (not optuna style)

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for x, y_age, y_gender in train_loader:

        x = x.to(DEVICE)
        y_age = y_age.to(DEVICE)
        y_gender = y_gender.to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            age_logits, gender_logits = model(x)

            loss_age = age_loss_fn(age_logits, y_age)
            loss_gender = gender_loss_fn(gender_logits, y_gender)

            loss = 0.7 * loss_age + 0.3 * loss_gender

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f}")

In [ ]:
model.eval()

correct_age = 0
correct_gender = 0
total = 0

with torch.no_grad():
    for x, y_age, y_gender in val_loader:

        x = x.to(DEVICE)

        age_logits, gender_logits = model(x)

        pred_age = age_logits.argmax(1).cpu()
        # pred_gender = gender_logits.argmax(1).cpu()
        gender_probs = torch.softmax(gender_logits, dim=1)

        gender_confidence, pred_gender = gender_probs.max(1)
        
        pred_gender = pred_gender.cpu()
        gender_confidence = gender_confidence.cpu()

        correct_age += (pred_age == y_age).sum().item()
        correct_gender += (pred_gender == y_gender).sum().item()

        total += y_age.size(0)

print("Age acc:", correct_age / total)
print("Gender acc:", correct_gender / total)

In [ ]:
test = np.load("test_embeddings.npz")

X_test = test["X"]
y_age_test = test["y_age"]
y_gender_test = test["y_gender"]

In [ ]:
test_loader = DataLoader(
    EmbeddingDataset(X_test, y_age_test, y_gender_test),
    batch_size=256,
    shuffle=False
)

In [ ]:
model.eval()

correct_age = 0
correct_gender = 0
total = 0

with torch.no_grad():

    for x, y_age, y_gender in test_loader:

        x = x.to(DEVICE)

        age_logits, gender_logits = model(x)

        pred_age = age_logits.argmax(1).cpu()
        # pred_gender = gender_logits.argmax(1).cpu()
        gender_probs = torch.softmax(gender_logits, dim=1)

        gender_confidence, pred_gender = gender_probs.max(1)
        
        pred_gender = pred_gender.cpu()
        gender_confidence = gender_confidence.cpu()

        correct_age += (pred_age == y_age).sum().item()
        correct_gender += (pred_gender == y_gender).sum().item()

        total += y_age.size(0)

print("FINAL TEST AGE ACC:", correct_age / total)
print("FINAL TEST GENDER ACC:", correct_gender / total)

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

In [ ]:
model.eval()

all_age_preds = []
all_age_true = []

all_gender_preds = []
all_gender_true = []

with torch.no_grad():

    for x, y_age, y_gender in test_loader:

        x = x.to(DEVICE)

        age_logits, gender_logits = model(x)

        pred_age = age_logits.argmax(1).cpu().numpy()
        # pred_gender = gender_logits.argmax(1).cpu().numpy()
        gender_probs = torch.softmax(gender_logits, dim=1)

        gender_confidence, pred_gender = gender_probs.max(1)
        
        pred_gender = pred_gender.cpu().numpy()
        gender_confidence = gender_confidence.cpu().numpy()

        all_age_preds.extend(pred_age)
        all_age_true.extend(y_age.numpy())

        all_gender_preds.extend(pred_gender)
        all_gender_true.extend(y_gender.numpy())

In [ ]:
print(
    classification_report(
        all_age_true,
        all_age_preds,
        target_names=AGE_LABELS
    )
)

In [ ]:
cm_age = confusion_matrix(all_age_true, all_age_preds)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_age,
    display_labels=AGE_LABELS
)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax)

plt.title("Age Group Confusion Matrix")
plt.show()

In [ ]:
GENDER_LABELS = ["Male", "Female"]

print(
    classification_report(
        all_gender_true,
        all_gender_preds,
        target_names=GENDER_LABELS
    )
)

In [ ]:
cm_gender = confusion_matrix(
    all_gender_true,
    all_gender_preds
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_gender,
    display_labels=GENDER_LABELS
)

fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax)

plt.title("Gender Confusion Matrix")
plt.show()

In [ ]:

class InferenceStatus(Enum):
    """Enum for inference result statuses."""
    
    SUCCESS = "SUCCESS"
    NO_FACE = "NO_FACE_DETECTED"
    MULTIPLE_FACES = "MULTIPLE_FACES_DETECTED"
    IMAGE_LOAD_FAILED = "IMAGE_LOAD_FAILED"


In [ ]:
def predict_image(img_path):
    """
    Predict age and gender from an image.
    
    Returns:
        dict: Status and prediction results with keys:
            - status (InferenceStatus): Result status
            - age_prediction (str): Age label (if SUCCESS)
            - age_confidence (float): Confidence score (if SUCCESS)
            - gender_prediction (str): Gender label (if SUCCESS)
            - gender_confidence (float): Confidence score (if SUCCESS)
    """
    
    img = cv2.imread(img_path)
    if img is None:
        return {"status": InferenceStatus.IMAGE_LOAD_FAILED}
    
    faces = app.get(img)
    
    # if len(faces) == 0:
    #     return {"status": InferenceStatus.NO_FACE}
    
    # if len(faces) > 1:
    #     return {"status": InferenceStatus.MULTIPLE_FACES}
    
    # embedding = faces[0].embedding

    if len(faces) == 0:
        return {"status": InferenceStatus.NO_FACE}

    # ✅ FIX: always take best face instead of rejecting multiple faces
    faces = sorted(faces, key=lambda x: x.det_score, reverse=True)
    face = faces[0]
    
    embedding = face.embedding
    
    x = torch.tensor(
        embedding,
        dtype=torch.float32
    ).unsqueeze(0).to(DEVICE)
    
    model.eval()
    
    with torch.no_grad():
        age_logits, gender_logits = model(x)
        
        # AGE
        age_probs = torch.softmax(age_logits, dim=1)
        age_confidence, pred_age = age_probs.max(1)
        
        # GENDER
        gender_probs = torch.softmax(gender_logits, dim=1)
        gender_confidence, pred_gender = gender_probs.max(1)
    
    # Move to CPU and convert to Python scalars
    pred_age_idx = pred_age.item()
    age_conf = age_confidence.item()
    pred_gender_idx = pred_gender.item()
    gender_conf = gender_confidence.item()
    
    # Print results for immediate feedback
    print("=" * 50)
    print("PREDICTED AGE:", AGE_LABELS[pred_age_idx])
    print("AGE CONFIDENCE:", round(age_conf, 4))
    print()
    print("PREDICTED GENDER:", "Male" if pred_gender_idx == 1 else "Female")
    print("GENDER CONFIDENCE:", round(gender_conf, 4))
    
    # Return structured response
    return {
        "status": InferenceStatus.SUCCESS,
        "age_prediction": AGE_LABELS[pred_age_idx],
        "age_confidence": age_conf,
        "gender_prediction": "Male" if pred_gender_idx == 1 else "Female",
        "gender_confidence": gender_conf
    }


In [ ]:
predict_image("/kaggle/input/datasets/biggonibbo/car-img/car.jpg")

In [ ]:
import os

folder = "/kaggle/input/datasets/biggonibbo/test-images/test_images"

for fname in os.listdir(folder):

    path = os.path.join(folder, fname)

    print("\nFILE:", fname)

    predict_image(path)

In [ ]:
import os
import cv2
import math
import matplotlib.pyplot as plt

folder = "/kaggle/input/datasets/biggonibbo/test-images/test_images"

image_paths = [
    os.path.join(folder, f)
    for f in os.listdir(folder)
]

cols = 3
rows = math.ceil(len(image_paths) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flatten()

for idx, img_path in enumerate(image_paths):

    ax = axes[idx]

    img = cv2.imread(img_path)

    if img is None:
        ax.set_title("LOAD FAILED")
        ax.axis("off")
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    result = predict_image(img_path)

    ax.imshow(img)
    ax.axis("off")

    # ❌ handle failure cases
    if result["status"] != InferenceStatus.SUCCESS:
        ax.set_title(result["status"].value, fontsize=10)
        continue

    # ✅ ONLY gender + confidence
    gender = result["gender_prediction"]
    conf = result["gender_confidence"]

    ax.set_title(
        f"{gender} ({conf:.2f})",
        fontsize=12
    )

# hide empty plots
for i in range(len(image_paths), len(axes)):
    axes[i].axis("off")

plt.tight_layout()

# save output
out_path = "/kaggle/working/gender_predictions.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")

plt.show()

print("Saved to:", out_path)